**Criando o notebook ‘02_transformacao_silver’:**

In [0]:
from pyspark.sql import functions as F

catalogo = "workspace"
schema_bronze = "ecommerce_bronze"
schema_silver = "ecommerce_silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{schema_silver}")

def limpar_textos(df):
    for coluna, tipo in df.dtypes:
        if tipo == "string":
            df = df.withColumn(
                coluna,
                F.when(F.trim(F.col(coluna)) == "", None)
                 .otherwise(F.trim(F.col(coluna)))
            )
    return df

clientes = (
    spark.table(f"{catalogo}.{schema_bronze}.customer_master")
    .transform(limpar_textos)
    .select(
        "customer_id",
        "customer_name",
        F.col("customer_age").cast("int").alias("customer_age"),
        "gender",
        "customer_segment",
        "customer_city",
        "customer_state",
        "customer_country",
        "region",
        "customer_postal_code",
        F.col("customer_acquisition_cost")
        .cast("decimal(18,2)")
        .alias("customer_acquisition_cost"),
        "_source_file",
        "_ingestion_timestamp"
    )
    .dropDuplicates(["customer_id"])
)

produtos = (
    spark.table(f"{catalogo}.{schema_bronze}.product_catalog")
    .transform(limpar_textos)
    .select(
        "product_id",
        "product_name",
        "product_category",
        "product_subcategory",
        "brand",
        "supplier",
        F.col("unit_price")
        .cast("decimal(18,2)")
        .alias("catalog_unit_price"),
        F.col("product_cost")
        .cast("decimal(18,2)")
        .alias("catalog_product_cost"),
        F.col("product_rating")
        .cast("decimal(3,2)")
        .alias("product_rating"),
        "_source_file",
        "_ingestion_timestamp"
    )
    .dropDuplicates(["product_id"])
)

pedidos = (
    spark.table(f"{catalogo}.{schema_bronze}.sales_orders")
    .transform(limpar_textos)
    .select(
        "order_id",
        F.to_date("order_date", "yyyy-MM-dd").alias("order_date"),
        F.to_timestamp(
            F.concat_ws(" ", F.col("order_date"), F.col("order_time")),
            "yyyy-MM-dd HH:mm:ss"
        ).alias("order_timestamp"),
        "order_status",
        "sales_channel",
        "customer_id",
        "customer_type",
        "payment_method",
        "payment_status",
        "currency",
        "shipping_method",
        "warehouse",

        F.expr("try_cast(delivery_days AS DOUBLE)")
        .cast("int")
        .alias("delivery_days"),

        F.expr("try_cast(estimated_delivery_days AS DOUBLE)")
        .cast("int")
        .alias("estimated_delivery_days"),

        "delivery_status",
        "return_status",
        "return_reason",

        F.expr("try_cast(customer_rating AS DOUBLE)")
        .cast("decimal(3,2)")
        .alias("customer_rating"),

        "review_sentiment",
        "customer_review",
        "marketing_channel",
        "campaign_name",
        "coupon_code",

        F.expr("try_cast(loyalty_points_earned AS DOUBLE)")
        .cast("int")
        .alias("loyalty_points_earned"),

        F.expr("try_cast(loyalty_points_redeemed AS DOUBLE)")
        .cast("int")
        .alias("loyalty_points_redeemed"),

        F.expr("try_cast(customer_lifetime_value AS DOUBLE)")
        .cast("decimal(18,2)")
        .alias("customer_lifetime_value"),

        F.expr("try_cast(is_repeat_customer AS BOOLEAN)")
        .alias("is_repeat_customer"),

        F.expr("try_cast(customer_order_count AS DOUBLE)")
        .cast("int")
        .alias("customer_order_count"),

        "_source_file",
        "_ingestion_timestamp"
    )
    .dropDuplicates(["order_id"])
)


itens_pedido = (
    spark.table(f"{catalogo}.{schema_bronze}.order_items")
    .transform(limpar_textos)
    .select(
        "order_id",
        "product_id",
        F.col("quantity").cast("int").alias("quantity"),
        F.col("unit_price").cast("decimal(18,2)").alias("unit_price"),
        F.col("discount_percentage")
        .cast("decimal(10,6)")
        .alias("discount_percentage"),
        F.col("discount_amount")
        .cast("decimal(18,2)")
        .alias("discount_amount"),
        F.col("gross_sales").cast("decimal(18,2)").alias("gross_sales"),
        F.col("tax_amount").cast("decimal(18,2)").alias("tax_amount"),
        F.col("shipping_cost")
        .cast("decimal(18,2)")
        .alias("shipping_cost"),
        F.col("net_sales").cast("decimal(18,2)").alias("net_sales"),
        F.col("product_cost").cast("decimal(18,2)").alias("product_cost"),
        F.col("profit").cast("decimal(18,2)").alias("profit"),
        "_source_file",
        "_ingestion_timestamp"
    )
    .dropDuplicates()
)

estatisticas = (
    spark.table(f"{catalogo}.{schema_bronze}.dataset_statistics")
    .transform(limpar_textos)
    .select(
        F.col("total_transactions").cast("int").alias("total_transactions"),
        F.col("total_columns").cast("int").alias("total_columns"),
        F.col("total_customers").cast("int").alias("total_customers"),
        F.col("total_products_used").cast("int").alias("total_products_used"),
        "date_range",
        F.regexp_replace("total_revenue", "[$,]", "")
        .cast("decimal(18,2)")
        .alias("total_revenue"),
        F.regexp_replace("total_profit", "[$,]", "")
        .cast("decimal(18,2)")
        .alias("total_profit"),
        F.regexp_replace("average_order_value", "[$,]", "")
        .cast("decimal(18,2)")
        .alias("average_order_value"),
        F.col("average_rating").cast("decimal(3,2)").alias("average_rating"),
        F.regexp_replace("return_rate", "%", "")
        .cast("decimal(5,2)")
        .alias("return_rate_percentage"),
        F.regexp_replace("cancellation_rate", "%", "")
        .cast("decimal(5,2)")
        .alias("cancellation_rate_percentage"),
        "_source_file",
        "_ingestion_timestamp"
    )
)

tabelas_silver = {
    "clientes": clientes,
    "produtos": produtos,
    "pedidos": pedidos,
    "itens_pedido": itens_pedido,
    "estatisticas_dataset": estatisticas
}

for nome_tabela, dataframe in tabelas_silver.items():
    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalogo}.{schema_silver}.{nome_tabela}")
    )

    print(f"Tabela Silver criada: {catalogo}.{schema_silver}.{nome_tabela}")
    print(f"Quantidade de registros: {dataframe.count()}")


Tabela Silver criada: workspace.ecommerce_silver.clientes
Quantidade de registros: 25000
Tabela Silver criada: workspace.ecommerce_silver.produtos
Quantidade de registros: 1175
Tabela Silver criada: workspace.ecommerce_silver.pedidos
Quantidade de registros: 138116
Tabela Silver criada: workspace.ecommerce_silver.itens_pedido
Quantidade de registros: 397569
Tabela Silver criada: workspace.ecommerce_silver.estatisticas_dataset
Quantidade de registros: 1


**Validando a criação da tabela pedidos:**

In [0]:
%sql
SELECT
    order_id,
    order_date,
    delivery_days,
    estimated_delivery_days,
    customer_rating,
    loyalty_points_earned,
    customer_order_count
FROM workspace.ecommerce_silver.pedidos
LIMIT 10;


order_id,order_date,delivery_days,estimated_delivery_days,customer_rating,loyalty_points_earned,customer_order_count
ORD-324410,2022-02-25,0,0,4.00,91,8
ORD-376647,2024-02-12,4,4,3.10,18,8
ORD-936652,2022-02-09,null,null,null,0,10
ORD-500817,2024-02-01,null,null,null,0,10
ORD-939335,2021-07-04,7,7,3.70,90,4
ORD-791388,2022-10-30,null,null,null,0,8
ORD-529833,2023-11-26,null,null,null,0,8
ORD-226032,2021-03-12,1,1,4.20,172,6
ORD-573045,2025-06-07,5,6,4.40,175,6
ORD-323108,2021-01-02,7,7,3.10,160,12


**Verificando as tabelas do notebook:**

In [0]:
%sql
SHOW TABLES IN workspace.ecommerce_silver;


database,tableName,isTemporary
ecommerce_silver,clientes,false
ecommerce_silver,estatisticas_dataset,false
ecommerce_silver,itens_pedido,false
ecommerce_silver,pedidos,false
ecommerce_silver,produtos,false


In [0]:
%sql
SELECT *
FROM workspace.ecommerce_silver.pedidos
LIMIT 10;


order_id,order_date,order_timestamp,order_status,sales_channel,customer_id,customer_type,payment_method,payment_status,currency,shipping_method,warehouse,delivery_days,estimated_delivery_days,delivery_status,return_status,return_reason,customer_rating,review_sentiment,customer_review,marketing_channel,campaign_name,coupon_code,loyalty_points_earned,loyalty_points_redeemed,customer_lifetime_value,is_repeat_customer,customer_order_count,_source_file,_ingestion_timestamp
ORD-324410,2022-02-25,2022-02-25T16:56:09.000Z,Completed,Mobile App,CUST-010810,Loyal,Credit Card,Paid,GBP,Same Day,WH-001,0,0,On Time,null,null,4.00,Positive,Decent quality for the price.,Email Marketing,null,null,91,36,10908.86,true,8,ecommerce_sales_customer_analytics_150k.csv,2026-09-12T23:12:26.304Z
ORD-376647,2024-02-12,2024-02-12T20:15:09.000Z,Completed,Mobile App,CUST-013477,Loyal,PayPal,Pending,USD,Standard,WH-014,4,4,On Time,null,null,3.10,Neutral,Average product. Could be better.,Referral,null,null,18,11,6930.20,true,8,ecommerce_sales_customer_analytics_150k.csv,2026-09-12T23:12:26.304Z
ORD-936652,2022-02-09,2022-02-09T00:09:34.000Z,Pending,Website,CUST-015807,Loyal,Credit Card,Failed,USD,Standard,WH-009,null,null,Cancelled,null,null,null,null,null,Google Ads,Google_Display,null,0,0,13908.44,true,10,ecommerce_sales_customer_analytics_150k.csv,2026-09-12T23:12:26.304Z
ORD-500817,2024-02-01,2024-02-01T14:38:41.000Z,Pending,Website,CUST-001795,Loyal,Buy Now Pay Later,Paid,AED,Standard,WH-018,null,null,Cancelled,null,null,null,null,null,Referral,null,null,0,0,7289.85,true,10,ecommerce_sales_customer_analytics_150k.csv,2026-09-12T23:12:26.304Z
ORD-939335,2021-07-04,2021-07-04T12:25:34.000Z,Completed,Marketplace,CUST-010650,Loyal,PayPal,Paid,USD,Standard,WH-005,7,7,On Time,null,null,3.70,Positive,Decent quality for the price.,Email Marketing,null,null,90,33,4052.86,true,4,ecommerce_sales_customer_analytics_150k.csv,2026-09-12T23:12:26.304Z
ORD-791388,2022-10-30,2022-10-30T06:54:08.000Z,Cancelled,Mobile App,CUST-006291,Loyal,Digital Wallet,Failed,USD,Economy,WH-008,null,null,Cancelled,null,null,null,null,null,Referral,Referral_Program,null,0,0,11678.64,true,8,ecommerce_sales_customer_analytics_150k.csv,2026-09-12T23:12:26.304Z
ORD-529833,2023-11-26,2023-11-26T19:38:09.000Z,Cancelled,Mobile App,CUST-021107,Loyal,Debit Card,Failed,USD,Standard,WH-009,null,null,Cancelled,null,null,null,null,null,Facebook Ads,null,null,0,0,11166.46,true,8,ecommerce_sales_customer_analytics_150k.csv,2026-09-12T23:12:26.304Z
ORD-226032,2021-03-12,2021-03-12T07:28:38.000Z,Completed,Website,CUST-010445,Loyal,Cash on Delivery,Paid,USD,Express,WH-001,1,1,On Time,null,null,4.20,Positive,Met my expectations. Would recommend.,Organic Search,null,null,172,165,5925.25,true,6,ecommerce_sales_customer_analytics_150k.csv,2026-09-12T23:12:26.304Z
ORD-573045,2025-06-07,2025-06-07T00:07:38.000Z,Completed,Social Media,CUST-006848,Loyal,PayPal,Paid,USD,Standard,WH-009,5,6,Early,null,null,4.40,Positive,Decent quality for the price.,Google Ads,Google_Shopping,null,175,120,8132.84,true,6,ecommerce_sales_customer_analytics_150k.csv,2026-09-12T23:12:26.304Z
ORD-323108,2021-01-02,2021-01-02T16:33:10.000Z,Completed,Mobile App,CUST-020322,Loyal,Bank Transfer,Paid,USD,Economy,WH-016,7,7,On Time,null,null,3.10,Neutral,Product has some issues but it works.,Organic Search,null,null,160,76,16832.53,true,12,ecommerce_sales_customer_analytics_150k.csv,2026-09-12T23:12:26.304Z


**Quantidade de registros:**

In [0]:
%sql
SELECT COUNT(*) AS total_clientes
FROM workspace.ecommerce_silver.clientes;


total_clientes
25000


In [0]:
%sql
SELECT COUNT(*) AS total_pedidos
FROM workspace.ecommerce_silver.pedidos;


total_pedidos
138116


In [0]:
%sql
SELECT COUNT(*) AS total_itens
FROM workspace.ecommerce_silver.itens_pedido;


total_itens
397569
